In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").getOrCreate()

In [ ]:
cust_df = spark.read.csv("customers.txt", header=True, inferSchema=True, sep="|")
prod_df = spark.read.csv("products.txt", header=True, inferSchema=True, sep="|")
ord_df = spark.read.csv("orders.txt", header=True, inferSchema=True, sep="|")

cust_df.createOrReplaceTempView("Customers")
prod_df.createOrReplaceTempView("Products")
ord_df.createOrReplaceTempView("Orders")

In [3]:
print(cust_df.columns)
print(prod_df.columns)
print(ord_df.columns)

['cust_id', 'name', 'gender', 'city', 'signup_date', 'credit_limit']
['product_id', 'product_name', 'category', 'price', 'stock']
['order_id', 'cust_id', 'product_id', 'order_date', 'quantity', 'payment_mode', 'order_status']


In [ ]:
# practice
# 1. Display customer name and city of customers who are from cities where more than one customer exists.

spark.sql(
    """
select name,city from Customers
          where city in (select city from customers group by city having count(*)>1)
"""
).show()

# above can be done using self join also
spark.sql(
    """
select c1.name,c1.city from customers c1 join customers c2 on c1.city=c2.city and c1.cust_id<>c2.cust_id
          """
).show()

In [ ]:
# 2. Display customer name and total number of Delivered orders they placed.
# Include only customers who placed more than 1 delivered order.

spark.sql(
    """
select c.name, count(o.order_id) as totalOrders
from customers c join orders o 
on c.cust_id=o.cust_id
where o.order_status='Delivered'
group by c.name
having count(o.order_id)>1
"""
).show()

+-----+-----------+
| name|totalOrders|
+-----+-----------+
|Tarun|          2|
+-----+-----------+



In [ ]:
# 3. Display product_name and price of products that were ordered more than once (total quantity across all orders > 1).

spark.sql(
    """
select p.product_name,p.price 
          from products p 
          join orders o 
          on p.product_id=o.product_id
          group by p.product_id,p.product_name,p.price 
          having sum(o.quantity)>1
          """
).show()

# same above qsn using subquery
spark.sql(
    """
select product_name,price 
          from products 
          where product_id in (
          select product_id from orders
          group by product_id 
          having sum(quantity)>1
          )
"""
).show()

In [10]:
print(cust_df.columns)
print(prod_df.columns)
print(ord_df.columns)

['cust_id', 'name', 'gender', 'city', 'signup_date', 'credit_limit']
['product_id', 'product_name', 'category', 'price', 'stock']
['order_id', 'cust_id', 'product_id', 'order_date', 'quantity', 'payment_mode', 'order_status']


In [11]:
# Display customer name and total amount spent
# Only for customers whose total spending is greater than the average spending of all customers.

spark.sql(
    """
SELECT c.name,
       SUM(o.quantity * p.price) AS total_spent
FROM customers c
JOIN orders o 
    ON c.cust_id = o.cust_id
JOIN products p 
    ON o.product_id = p.product_id
GROUP BY c.cust_id, c.name
HAVING SUM(o.quantity * p.price) >
(
    SELECT AVG(customer_total)
    FROM (
        SELECT SUM(o2.quantity * p2.price) AS customer_total
        FROM orders o2
        JOIN products p2
            ON o2.product_id = p2.product_id
        GROUP BY o2.cust_id
    ) AS sub
)
"""
).show()

+-----+-----------+
| name|total_spent|
+-----+-----------+
|Tarun|      84000|
|Aisha|      60000|
|Imran|      30000|
| Sara|      80000|
+-----+-----------+

